# Sprint 0 — Owner A (Foundations)

Safe TSV I/O, normalisation, rule lists, placeholder detection and the proxy
data generator for the Amazon ML Challenge 2026 Business Entity Resolution build.

Specs: `docs/io_rules.md` (binding I/O rules), `docs/research.md` §3.1 / §10,
task list in `sprints/sprint-0/owner-A-tasks.md`.

CPU-only — no GPU runtime needed for Owner A.

## 1. Setup

Set `REPO_URL` to your remote, or skip the clone if you mounted Drive.

In [ ]:
import os, sys, subprocess
from pathlib import Path

REPO_URL = ""  # e.g. "https://github.com/<you>/goin-bezobserk.git"
REPO_DIR = Path("/content/goin-bezobserk")
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    if REPO_URL and not REPO_DIR.exists():
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
    if REPO_DIR.exists():
        os.chdir(REPO_DIR)
else:
    # running locally from notebooks/
    if Path.cwd().name == "notebooks":
        os.chdir(Path.cwd().parent)

sys.path.insert(0, str(Path.cwd() / "src"))
print("cwd:", Path.cwd())

In [ ]:
# `metaphone` is optional: normalise.double_metaphone falls back to a built-in
# reduction when it is missing, so the pipeline never hard-depends on it.
try:
    import metaphone  # noqa: F401
    print("metaphone available")
except ImportError:
    if IN_COLAB:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "metaphone"], check=False)
    print("metaphone not installed - using built-in fallback")

In [ ]:
import pandas as pd
from ber import audit, normalise, placeholders, proxy_data, safe_io, run_audit

print("pandas", pd.__version__)
print("A2.1 contract fields:")
for field in normalise.NORMALISED_FIELDS:
    print("   ", field)

## 2. A1 — the safe reader and the three sanity checks

`io_rules.md` §1. The flags are not negotiable; §8 lists reading without them
as a silent killer. The checks **raise** rather than warn (A1.3).

In [ ]:
import inspect
print(inspect.getsource(safe_io.read_source))

In [ ]:
# Proof the NA trap is closed: a business literally named "NA" survives.
demo = Path("data/interim/demo"); demo.mkdir(parents=True, exist_ok=True)
(demo / "s1.tsv").write_text(
    "entity_id\tbusiness_name\tbusiness_address\tcountry\n"
    "S1-00001\tNA\t12 Maple Street, Fairhaven 94105\tUS\n"
    "S1-00002\tNone\t\tIndia\n"
    "S1-00003\tCaf\u00e9 Lumi\u00e8re SARL\t2 Rue \u00c9mile Zola, Valmont 75008\tFrance\n",
    encoding="utf-8", newline="",
)

df, report = safe_io.load_source(demo / "s1.tsv")
print("sanity ok:", report.ok, "| rows:", report.n_rows, "| NaN:", report.n_nan)
df

In [ ]:
# And proof the guard fires on a malformed row (check 2: field count).
(demo / "bad.tsv").write_text(
    "entity_id\tbusiness_name\tbusiness_address\tcountry\n"
    "S1-00001\tA\t1 Maple Street\tUS\n"
    "S1-00002\tB\textra\ttab\tIndia\n",
    encoding="utf-8", newline="",
)
try:
    safe_io.load_source(demo / "bad.tsv")
except safe_io.SanityCheckError as exc:
    print("raised as designed:\n", exc)

## 3. A2/A3 — normalisation and the rule lists

`research.md` §3.1. Note that accents are **kept** alongside a folded copy, and
legal suffixes are returned as a **feature** rather than silently deleted (A3.2).

In [ ]:
examples = [
    ("Brightleaf Traders Pvt Ltd", "5 Gandhi Marg, Nashik 422001", "India"),
    ("Soci\u00e9t\u00e9 G\u00e9n\u00e9rale SA", "2 Rue \u00c9mile Zola, Valmont 75008", "France"),
    ("Acme Hardware LLC", "120 Maple St, Ste 200, Fairhaven 94105", "US"),
    ("Acme Hardware LLP", "120 Maple St, Ste 210, Fairhaven 94105", "US"),
    ("Shree Ganesh Traders", "opp bus stand, Indore 452001", "India"),
    ("NA", "-", "Atlantis"),
]

rows = []
for name, addr, country in examples:
    r = normalise.normalise_record(name, addr, country)
    rows.append({
        "raw_name": name,
        "name_core": r.name_core,
        "ascii": r.name_core_ascii,
        "suffixes": ",".join(sorted(r.legal_suffixes)) or "-",
        "postcode": r.postcode or "-",
        "house": (r.addr_numbers or {}).get("house_number") or "-",
        "units": ",".join((r.addr_numbers or {}).get("units", [])) or "-",
        "landmark": r.landmark_flag,
        "phonetic": r.phonetic_key,
        "missing": f"name={r.name_missing},addr={r.addr_missing}",
    })
pd.DataFrame(rows)

Two things to read off that table:

- `Acme Hardware LLC` vs `LLP` share a `name_core` but differ in `suffixes` —
  that is the conflict signal Owner C needs (C3.6), not a deletion.
- `Ste 200` vs `Ste 210` share a house number but differ in `units` — the
  parsed-field comparison `research.md` §5 asks for, since fuzzy ratio is ~0.95.

## 4. A4 — placeholder detector

`io_rules.md` §4. The hard rule: two records must never match **because** they
share a placeholder.

In [ ]:
probe = ["", "-", "0", "NA", "None", "null", "n/a", "--", "000",
         "NA Foods", "Zero Degrees Cafe", "A1 Traders"]
pd.DataFrame({
    "value": probe,
    "is_placeholder": [placeholders.is_placeholder(v) for v in probe],
})

In [ ]:
# Two placeholder addresses must share no usable signal at all.
a = normalise.normalise_record("Alpha Traders", "NA", "India")
b = normalise.normalise_record("Beta Traders", "-", "India")
print("addr_norm:", repr(a.addr_norm), repr(b.addr_norm))
print("postcode :", a.postcode, b.postcode)
print("flags    :", a.addr_missing, b.addr_missing)

## 5. Input data — Google Drive mount

The organiser ships `student_resource/` with this layout:

```
student_resource/
  dataset/
    train/  train_source{1,2,3}.tsv  train_ground_truth.tsv
    test/   test_source{1,2,3}.tsv
  utils/validate_submission.py
  README.md  Documentation_template.md
```

Set `DRIVE_DATA_ROOT` to wherever you dropped `student_resource` (or its
`dataset/` dir) in Drive. `find_dataset_root` accepts either. If nothing is
found the notebook falls back to **proxy data**, so every cell below still runs
before the real dataset lands.

Raw data is read-only here — `CLAUDE.md` forbids modifying it in place.

In [ ]:
DRIVE_DATA_ROOT = "/content/drive/MyDrive/amzn-ml-challenge-26/student_resource"

if IN_COLAB:
    try:
        from google.colab import drive
        if not Path("/content/drive").exists():
            drive.mount("/content/drive")
    except Exception as exc:
        print("Drive mount skipped:", exc)

# also try a repo-local dataset/ dir, for running outside Colab
DATA_ROOT = run_audit.find_dataset_root(DRIVE_DATA_ROOT, "dataset", "student_resource")
USE_REAL_DATA = DATA_ROOT is not None
print("dataset root:", DATA_ROOT if USE_REAL_DATA else "NOT FOUND -> falling back to proxy data")
if USE_REAL_DATA:
    for split in ("train", "test"):
        print(f"  {split}:", sorted(q.name for q in (DATA_ROOT / split).glob("*.tsv")))

In [ ]:
if USE_REAL_DATA:
    train_frames, train_reports = run_audit.load_dataset(DATA_ROOT / "train")
    test_frames, test_reports = run_audit.load_dataset(DATA_ROOT / "test")
    train_truth = run_audit.load_ground_truth(DATA_ROOT / "train")
    truth = run_audit.load_ground_truth(DATA_ROOT / "test")  # usually absent
    for phase, reports in (("train", train_reports), ("test", test_reports)):
        for source, rep in reports.items():
            print(f"{phase} {source}: rows={rep.n_rows} nan={rep.n_nan} ok={rep.ok}")
    print("train truth S1 rows:", len(train_truth) if train_truth else 0)
else:
    split, paths = run_audit.build_proxy(
        "data/interim/proxy", n_entities=400, seed=20260925, singleton_rate=0.40
    )
    train_frames, _ = run_audit.load_dataset("data/interim/proxy/train")
    test_frames, _ = run_audit.load_dataset("data/interim/proxy/test")
    train_truth = split["train"][3]
    truth = split["test"][3]

print({k: len(v) for k, v in test_frames.items()})
test_frames["S2"].head(8)

## 6. Full audit (research.md §10)

This is the report Owner A hands to B, C and D. The `[VERIFY]` probes at the
bottom are what unblock D5.3 (bipartite N2) and B0.1 (country sharding).

In [ ]:
# On real data the test split has no ground truth, so the Q3/Q4/singleton
# probes run against TRAIN, and the test split gets the parsing/namespace/
# country/script half of the checklist plus the unseen-label diff.
if USE_REAL_DATA:
    run_audit._rule("TRAIN SPLIT")
    run_audit.run(train_frames, truth=train_truth)
    run_audit._rule("TEST SPLIT (France expected here, absent from train)")
    run_audit.run(test_frames, truth=truth, train_frames=train_frames)
else:
    run_audit.run(test_frames, truth=truth, train_frames=train_frames)

## 7. Normalise a whole frame — the handoff to B and C

`normalise_frame` joins the A2.1 contract columns onto the source frame,
leaving `entity_id` untouched (`io_rules.md` §2 forbids stripping the prefix).

In [ ]:
s1_norm = normalise.normalise_frame(test_frames["S1"])
print("columns:", list(s1_norm.columns))
s1_norm[["entity_id", "name_core", "name_core_ascii", "postcode", "country_norm", "script_flag"]].head(10)

In [ ]:
# A2.3 token IDF over S1 names -> feeds B4.1 (rare-token key) and C2 (IDF features)
idf, df_counts = audit.token_idf(s1_norm["name_tokens"])
common = audit.high_df_tokens(df_counts, len(s1_norm))[:10]
rare = sorted(idf.items(), key=lambda kv: -kv[1])[:10]
print("most common name tokens (down-weight by IDF, do NOT delete):")
for token, count, share in common:
    print(f"   {token:<16} df={count:<5} {share:.1%}")
print("\nrarest tokens (best blocking keys):")
for token, value in rare:
    print(f"   {token:<16} idf={value:.3f}")

## 8. Tests

48 tests, each pinned to a rule in `io_rules.md` §8 or an edge case in
`research.md` §5.

In [ ]:
subprocess.run([sys.executable, "-m", "pytest", "tests/test_owner_a.py", "-q"])

## Status

Done: A1 (reader + 3 checks + namespace/country audit), A2 (normaliser contract),
A3 (US/IN/FR rule lists), A4 (placeholder detector), A5 (proxy generator).

Still blocking, and **not** code (see `owner-A-tasks.md` §A0):

| # | Question | Status |
|---|---|---|
| A0.1 | Is libpostal / a gazetteer parser "external data"? | unresolved → regex-only, default excluded |
| A0.2 | Fragment → ≤1 S1? (Q3) | holds on proxy; re-run on real train |
| A0.3 | Within-vendor 1:1? (Q4) | does **not** hold on proxy → D keeps bipartite N2 off |
| A0.4 | Real singleton/orphan rates (Q5) | unknown → stays parameterised |
| A0.5 | Exact country strings | unknown → casing normalised, label kept |
| A0.6 | Non-Latin script census | proxy is Latin+accents only |

Next: Owner D's scorer and validator (two of the three Gate 0 criteria).